# tulip tutorial: train, evaluate, and predict

This notebook trains a Polish dialect classifier end to end on the in-memory
`synthetic` corpus, so it runs on a core install with **no downloads and no
heavy extras**.

> The synthetic corpus is a generated fixture, not real speech. It exists to
> demonstrate the API and machinery, not to make a dialectology claim. Point the
> pipeline at a real corpus (see `docs/datasets.md`) for meaningful numbers.

In [ ]:
from tulip.config.schemas import SplitConfig
from tulip.data import SyntheticSpec, generate_corpus, speaker_disjoint_split
from tulip.pipeline import DialectClassifier, evaluate_samples

## 1. Build a corpus

`generate_corpus` produces short, learnable, dialect-correlated texts with
several speakers per class, so a speaker-disjoint split is meaningful.

In [ ]:
samples = generate_corpus(SyntheticSpec(n_speakers_per_dialect=8, samples_per_speaker=12))
print(f"{len(samples)} samples across {len({s.labels.dialect for s in samples})} dialects")

## 2. Split it, speaker-disjoint

No speaker appears in more than one of train/validation/test, so a score
cannot reward memorising a voice. The split is stratified by dialect and seeded.

In [ ]:
splits = speaker_disjoint_split(samples, SplitConfig(seed=42))
print(f"train={len(splits.train)}  validation={len(splits.validation)}  test={len(splits.test)}")

## 3. Train a classifier

`DialectClassifier` composes feature extractors and a model into one trainable
object. Here: character TF-IDF into logistic regression.

In [ ]:
classifier = DialectClassifier(
    model="logistic_regression", features=["char_tfidf"], seed=42
).fit(splits.train)

## 4. Evaluate on the held-out test split

In [ ]:
report = evaluate_samples(classifier, splits.test, name="test")
print(report.to_markdown())

## 5. Classify a fresh sentence

A `Prediction` carries the full class distribution, not just a label.

In [ ]:
prediction = classifier.predict("hej baca kaj se owce pasą na holi")
print("predicted dialect:", prediction.label)
for candidate in prediction.probabilities[:3]:
    print(f"  {candidate.label:<12} {candidate.probability:.1%}")

## Next steps

- Compare several models on one frozen split: `examples/compare_models.py`.
- Acquire a real corpus and point a config at it: `docs/datasets.md`.
- Regenerate the reproducible leaderboard: `tulip leaderboard benchmarks/suite.yaml`.
- Explore the CLI: `tulip --help`.